# Day 8 · My First PD Model

*Work log: a cleaned version of my working notebook. The polished analysis lives in [`../08_pd_model_logistic_regression`](../08_pd_model_logistic_regression.ipynb).*

**Goals:** split the Day 7 WOE table into train / validation / out-of-time samples, fit a logistic regression, and evaluate it with AUC, Gini and KS against LendingClub's grade.

**Key results:** out-of-time AUC 0.655, Gini 0.31, KS 0.22, about 91% of grade's Gini (0.34). No overfitting (train 0.649 vs. validation 0.647). All coefficients are negative, as expected. Calibration drifted: predicted PD 14.2% vs. actual 14.7% out of time.

## 1. Setup
scikit-learn was installed once with `pip3 install scikit-learn` (then the kernel was restarted).

In [1]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA = (Path.cwd().parents[1] / "Data").resolve()   # project_root/Data
con = duckdb.connect()

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
import numpy as np
import sklearn
print("sklearn", sklearn.__version__)

sklearn 1.9.1


In [2]:
# Rebuild the analysis views (DuckDB views are in-memory and reset every session)

# loans: every accepted loan
con.sql(f"""
CREATE OR REPLACE VIEW loans AS
SELECT *
FROM '{DATA}/accepted.parquet'
""")

# loans_clean: drop the 33 file-footer rows and parse the text issue date ('Dec-2015') into a real date
con.sql("""
CREATE OR REPLACE VIEW loans_clean AS
SELECT *,
       strptime(issue_d, '%b-%Y') AS issue_date
FROM loans
WHERE loan_status IS NOT NULL
""")

# cohort: 36-month loans issued 2014-2015, all matured by Dec-2018 so outcomes are final
con.sql("""
CREATE OR REPLACE VIEW cohort AS
SELECT *
FROM loans_clean
WHERE issue_date BETWEEN '2014-01-01' AND '2015-12-31'
  AND TRIM(term) = '36 months'
""")

# rejected: declined applications
con.sql(f"""
CREATE OR REPLACE VIEW rejected AS
SELECT *
FROM '{DATA}/rejected.parquet'
""")

con.sql("SHOW TABLES").df()

,name
0,cohort
1,loans
2,loans_clean
3,rejected


## 2. Load the WOE table and split the samples

In [3]:
from sklearn.model_selection import train_test_split

# The WOE table saved on Day 7 (DuckDB reads parquet files directly)
model = con.sql(f"SELECT * FROM '{DATA}/model_woe.parquet'").df()

# Drop the 86 indeterminate loans: no outcome, so they can't be used for training
model = model[model["bad_flag"].notna()]

# The 8 WOE feature columns
woe_cols = [c for c in model.columns if c.endswith("_woe")]
print(woe_cols)

# Split by time: development vs. out-of-time
dev = model[model["issue_date"] < "2015-07-01"]
oot = model[model["issue_date"] >= "2015-07-01"]

# Within development: random 80% train / 20% validation, stratified to keep the bad rate equal
train, val = train_test_split(dev, test_size=0.2, random_state=42, stratify=dev["bad_flag"])

for name, d in [("train", train), ("val", val), ("oot", oot)]:
    print(f"{name:5s}  n = {len(d):>7,}   bad rate = {d['bad_flag'].mean():.2%}")

['fico_range_low_woe', 'annual_inc_woe', 'dti_woe', 'inq_last_6mths_woe', 'home_ownership_woe', 'verification_status_woe', 'purpose_woe', 'emp_length_woe']
train  n = 226,688   bad rate = 14.32%
val    n =  56,672   bad rate = 14.32%
oot    n = 162,297   bad rate = 14.74%


The out-of-time bad rate is higher (14.74% vs. 14.32%). On Day 6 this turned out to be **population drift**: in 2015, borrowers with the same profile defaulted more, rather than the applicant mix getting worse.

## 3. Logistic regression
`z = intercept + Σ coefficient × WOE`, then `PD = 1 / (1 + e^(−z))`. Training finds the coefficients that best match the observed outcomes.

In [4]:
from sklearn.linear_model import LogisticRegression

X_train = train[woe_cols]          # convention: X = input features (a table)
y_train = train["bad_flag"]        #             y = the outcome to predict (a column)

lr = LogisticRegression(max_iter=1000)    # max_iter: maximum optimisation steps
lr.fit(X_train, y_train)                  # fit = train the model

coef = pd.DataFrame({"feature": woe_cols, "coefficient": lr.coef_[0]})
print("intercept:", lr.intercept_[0].round(3))
coef.round(3)

intercept: -1.795


,feature,coefficient
0,fico_range_low_woe,-0.832
1,annual_inc_woe,-0.595
2,dti_woe,-0.764
3,inq_last_6mths_woe,-0.730
4,home_ownership_woe,-0.659
5,verification_status_woe,-0.553
6,purpose_woe,-0.839
7,emp_length_woe,-0.403


* **All eight coefficients are negative.** A higher WOE means a safer bin, so it must lower the predicted PD. A positive sign would signal multicollinearity, a redundant feature or a non-monotonic binning.
* Intercept −1.795: when every WOE is 0 (average on everything), PD = 1 / (1 + e^1.795) = 14.2%, essentially the average bad rate.

## 4. AUC on all three samples
**AUC** = the probability that a randomly chosen bad borrower gets a higher predicted PD than a randomly chosen good one (0.5 = random, 1.0 = perfect).

In [5]:
from sklearn.metrics import roc_auc_score

for name, d in [("train", train), ("val", val), ("oot", oot)]:
    pd_hat = lr.predict_proba(d[woe_cols])[:, 1]       # column 1 = probability of default
    auc = roc_auc_score(d["bad_flag"], pd_hat)
    print(f"{name:5s}  AUC = {auc:.4f}   avg predicted PD = {pd_hat.mean():.2%}   actual bad rate = {d['bad_flag'].mean():.2%}")

train  AUC = 0.6489   avg predicted PD = 14.32%   actual bad rate = 14.32%
val    AUC = 0.6468   avg predicted PD = 14.28%   actual bad rate = 14.32%


oot    AUC = 0.6551   avg predicted PD = 14.18%   actual bad rate = 14.74%


* **No overfitting:** train 0.649 vs. validation 0.647.
* **Discrimination vs. calibration:** OOT AUC holds (0.655), so the ranking still works. But the model under-predicts the OOT bad rate (14.18% vs. 14.74%): the **PD level** drifted. Fix: recalibrate the intercept on recent data rather than rebuild.

## 5. Gini and KS
* **Gini** = 2 × AUC − 1 (and AUC = (Gini + 1) / 2).
* **KS**: sort borrowers by PD; the largest gap between the cumulative share of bads and of goods. Rough guide: < 0.2 weak, 0.2–0.3 fair, 0.3–0.4 good, > 0.75 suspect leakage.

In [6]:
from sklearn.metrics import roc_curve

def ks_stat(y, p):
    """KS = maximum gap between the cumulative share of bads and of goods."""
    fpr, tpr, _ = roc_curve(y, p)    # tpr: cumulative share of bads; fpr: of goods
    return (tpr - fpr).max()

results = []
for name, d in [("train", train), ("val", val), ("oot", oot)]:
    pd_hat = lr.predict_proba(d[woe_cols])[:, 1]
    auc = roc_auc_score(d["bad_flag"], pd_hat)
    results.append({
        "sample": name,
        "AUC": auc,
        "Gini": 2 * auc - 1,
        "KS": ks_stat(d["bad_flag"], pd_hat),
        "avg_PD": pd_hat.mean(),
        "bad_rate": d["bad_flag"].mean(),
    })

metrics = pd.DataFrame(results)
metrics.round(4)

,sample,AUC,Gini,KS,avg_PD,bad_rate
0,train,0.6489,0.2979,0.2149,0.1432,0.1432
1,val,0.6468,0.2937,0.2120,0.1428,0.1432
2,oot,0.6551,0.3102,0.2223,0.1418,0.1474


## 6. Benchmark against LendingClub's grade

In [7]:
oot_grade = con.sql("""
SELECT grade,
       CASE WHEN loan_status IN ('Charged Off', 'Late (31-120 days)') THEN 1
            WHEN loan_status = 'Fully Paid' THEN 0 END AS bad_flag
FROM cohort
WHERE issue_date >= '2015-07-01'
""").df()
oot_grade = oot_grade[oot_grade["bad_flag"].notna()]

# Letters to a risk order: A = 0 (safest) ... G = 6 (riskiest)
grade_map = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4, "F": 5, "G": 6}
oot_grade["grade_score"] = oot_grade["grade"].map(grade_map)

auc_g = roc_auc_score(oot_grade["bad_flag"], oot_grade["grade_score"])
grade_row = {"sample": "oot (grade)", "AUC": auc_g, "Gini": 2 * auc_g - 1,
             "KS": ks_stat(oot_grade["bad_flag"], oot_grade["grade_score"])}

compare = pd.concat([metrics[metrics["sample"] == "oot"], pd.DataFrame([grade_row])])
compare[["sample", "AUC", "Gini", "KS"]].round(4)

,sample,AUC,Gini,KS
2,oot,0.6551,0.3102,0.2223
0,oot (grade),0.6700,0.3400,0.2660


**Findings**
* Grade wins: Gini 0.34 vs. 0.31 (KS 0.27 vs. 0.22). Grade is built on the full credit report, platform data, years of history and possibly more complex models; this model uses eight public application fields.
* Even so, the model reaches **0.31 / 0.34 ≈ 91%** of grade's Gini with a simple, stable, explainable model.
* In champion / challenger terms, the challenger has not won yet. Next: more bureau features (credit history length, account counts, public records), better binning for revolving utilization, a gradient-boosting challenger, and intercept recalibration.